# Actinic keratosis — HAM10000 pipeline (Grad-CAM fixed)

**What changed from the original notebook**
1. `pip install grad-cam` instead of `pytorch-grad-cam` (the old name doesn't exist on PyPI, which is why Grad-CAM crashed). The current grad-cam version also requires a `targets` argument, so the Grad-CAM call now passes `targets=None` (= explain the predicted class).
2. Seeds set (42) and the trained model saved, so the heatmaps can be traced to one specific model.
3. **One stratified split for all four models.** The CNN previously used the Hugging Face splitter, which is not stratified, so its test set differed from the classical models' test set. Now one stratified 80:20 split (`random_state=42`, 20 test images per class) is computed once and shared by the CNN, Random Forest, SVM and XGBoost.
4. Single Grad-CAM figure saved as `gradcam_single.png`, plus a new panel cell (cell 2): Grad-CAM for the first 2 test images of every class → `gradcam_panel.png` + `gradcam_panel_predictions.csv`.

Cells 1 and 2 are the whole pipeline. The original submitted notebook is kept separately in the repository as `Actinic_keratosis.ipynb`.

**How to run:** Runtime → Change runtime type → **T4 GPU**. Then run **cell 1** and **cell 2** (about 5–10 min).

**Note:** the original run set no seed and used a different (unstratified) test split, so this retrains a fresh model on the shared stratified split. Its accuracy may differ from the 0.700 reported originally. Report the accuracy from the same run as the heatmaps.


In [ ]:
# ============================================================
# GOOGLE COLAB PIPELINE
# HAM10000 (Hugging Face) + ML + CNN + Explainable AI
# ============================================================

# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

# FIX: the PyPI package is 'grad-cam' (import name stays pytorch_grad_cam)
!pip install -q datasets transformers timm grad-cam xgboost

# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os

from PIL import Image

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

from tqdm import tqdm

# ---- ADDED: reproducibility (the original run set no seed) ----
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ============================================================
# 3. LOAD HAM10000 DATASET FROM HUGGING FACE
# ============================================================

dataset = load_dataset("pranay-43/HAM10000")

print(dataset)

# ============================================================
# 4. VIEW SAMPLE
# ============================================================

sample = dataset['train'][0]

print(sample)

# ============================================================
# 5. CLASS LABELS
# ============================================================

labels = dataset['train']['label']

print("Unique labels:", set(labels))

# ============================================================
# 6. LABEL MAPPING
# ============================================================

label_map = {
    0: "akiec",
    1: "bcc",
    2: "bkl",
    3: "df",
    4: "mel",
    5: "nv",
    6: "vasc"
}

# ============================================================
# 7. VISUALIZE SAMPLE IMAGES
# ============================================================

plt.figure(figsize=(12,8))

for i in range(6):

    image = dataset['train'][i]['image']
    label = dataset['train'][i]['label']

    plt.subplot(2,3,i+1)
    plt.imshow(image)
    plt.title(label_map[label])
    plt.axis("off")

plt.tight_layout()
plt.show()

# ============================================================
# 8. IMAGE TRANSFORMATIONS
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# ============================================================
# 9. CUSTOM DATASET CLASS
# ============================================================

class HAMDataset(Dataset):

    def __init__(self, hf_dataset, transform=None):

        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        image = self.dataset[idx]['image']
        label = self.dataset[idx]['label']

        image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# ============================================================
# 10. TRAIN TEST SPLIT
# ============================================================

# FIX: one stratified split shared by all four models. The CNN previously
# used the Hugging Face splitter, which is not stratified, so its test set
# differed from the classical models' test set.
all_labels = dataset['train']['label']

train_idx, test_idx = train_test_split(
    np.arange(len(all_labels)),
    test_size=0.2,
    random_state=42,
    stratify=all_labels
)

train_dataset = HAMDataset(dataset['train'].select(train_idx), transform)
test_dataset  = HAMDataset(dataset['train'].select(test_idx), transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ============================================================
# 11. LOAD PRETRAINED EFFICIENTNET MODEL
# ============================================================

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

num_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(num_features, 7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

# ============================================================
# 12. LOSS + OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ============================================================
# 13. TRAIN CNN MODEL
# ============================================================

epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {running_loss:.4f}")

# ---- ADDED: keep the exact model that produces the heatmaps ----
torch.save(model.state_dict(), "efficientnet_b0_ham700_seed42.pt")
print("Saved efficientnet_b0_ham700_seed42.pt")

# ============================================================
# 14. EVALUATE CNN MODEL
# ============================================================

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

# ============================================================
# 15. CNN PERFORMANCE
# ============================================================

acc = accuracy_score(y_true, y_pred)

print("\nCNN Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# ============================================================
# 16. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("CNN Confusion Matrix")

plt.show()

# ============================================================
# 17. FEATURE EXTRACTION FOR CLASSICAL ML
# ============================================================

def extract_features(dataset, size=(64,64)):

    X = []
    y = []

    for item in dataset:

        image = item['image']

        image = image.resize(size)

        image = np.array(image)

        image = image.flatten()

        X.append(image)

        y.append(item['label'])

    return np.array(X), np.array(y)

X, y = extract_features(dataset['train'])

# ============================================================
# 18. SPLIT DATA
# ============================================================

# same split as the CNN: reuse the stratified indices from section 10,
# so all four models are evaluated on identical test images
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# ============================================================
# 19. RANDOM FOREST
# ============================================================

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

print("\nRandom Forest Accuracy:")
print(accuracy_score(y_test, rf_preds))

# ============================================================
# 20. SVM
# ============================================================

svm = SVC(kernel='rbf')

svm.fit(X_train, y_train)

svm_preds = svm.predict(X_test)

print("\nSVM Accuracy:")
print(accuracy_score(y_test, svm_preds))

# ============================================================
# 21. XGBOOST
# ============================================================

xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=7
)

xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_test)

print("\nXGBoost Accuracy:")
print(accuracy_score(y_test, xgb_preds))

# ============================================================
# 22. GRAD-CAM EXPLAINABLE AI
# ============================================================

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

target_layers = [model.features[-1]]

cam = GradCAM(
    model=model,
    target_layers=target_layers
)

# ============================================================
# 23. TEST IMAGE FOR GRAD-CAM
# ============================================================

image, label = test_dataset[0]

input_tensor = image.unsqueeze(0).to(device)

# FIX: grad-cam >= 1.5 requires `targets`; None = use the predicted class
grayscale_cam = cam(input_tensor=input_tensor, targets=None)

grayscale_cam = grayscale_cam[0]

# ============================================================
# 24. VISUALIZATION
# ============================================================

rgb_img = image.permute(1,2,0).numpy()

visualization = show_cam_on_image(
    rgb_img,
    grayscale_cam,
    use_rgb=True
)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(rgb_img)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(visualization)
plt.title("Grad-CAM Heatmap")
plt.axis("off")

plt.savefig("gradcam_single.png", dpi=300, bbox_inches="tight")  # ADDED
plt.show()

# ============================================================
# 25. FINAL MODEL COMPARISON
# ============================================================

results = pd.DataFrame({

    "Model": [
        "Random Forest",
        "SVM",
        "XGBoost",
        "EfficientNet CNN"
    ],

    "Accuracy": [
        accuracy_score(y_test, rf_preds),
        accuracy_score(y_test, svm_preds),
        accuracy_score(y_test, xgb_preds),
        acc
    ]
})

print(results)

# ============================================================
# END OF PIPELINE
# ============================================================

In [ ]:
# ============================================================
# 22b. ADDED — GRAD-CAM PANEL: FIRST 2 TEST IMAGES OF EACH CLASS
# Fixed rule for picking images (not cherry-picked), so the panel
# can be shown honestly, including wrong predictions.
# Run this straight after the cell above (same runtime).
# ============================================================

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

model.eval()
cam = GradCAM(model=model, target_layers=[model.features[-1]])

test_labels = [all_labels[i] for i in test_idx]

fig, axes = plt.subplots(7, 4, figsize=(12, 21))
for ax in axes.ravel():
    ax.axis("off")

rows = []
for c in range(7):
    idxs = [i for i, l in enumerate(test_labels) if l == c][:2]
    for j, idx in enumerate(idxs):
        image, label = test_dataset[idx]
        x = image.unsqueeze(0).to(device)

        with torch.no_grad():
            probs = torch.softmax(model(x), dim=1)[0]
        pred = int(probs.argmax())
        conf = float(probs[pred])

        # heatmap for the class the model predicted
        heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(pred)])[0]
        rgb = image.permute(1, 2, 0).cpu().numpy()
        overlay = show_cam_on_image(rgb, heat, use_rgb=True)

        correct = pred == label
        axes[c, 2 * j].imshow(rgb)
        axes[c, 2 * j].set_title(f"True: {label_map[label]}", fontsize=10)
        axes[c, 2 * j + 1].imshow(overlay)
        axes[c, 2 * j + 1].set_title(
            f"Pred: {label_map[pred]} ({conf:.2f}) {'correct' if correct else 'WRONG'}",
            fontsize=10,
            color="green" if correct else "red",
        )
        rows.append({"test_index": idx, "true": label_map[label],
                     "pred": label_map[pred], "confidence": round(conf, 3),
                     "correct": correct})

plt.suptitle("Grad-CAM — EfficientNet-B0, first 2 test images per class",
             fontsize=14, weight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("gradcam_panel.png", dpi=300, bbox_inches="tight")
plt.show()

panel_df = pd.DataFrame(rows)
panel_df.to_csv("gradcam_panel_predictions.csv", index=False)
print(panel_df.to_string(index=False))

print(f"\nCNN accuracy for THIS run: {acc:.3f}")
print("If this differs from 0.700, report the numbers from this run alongside these heatmaps.")
print("Check each heatmap yourself: is the red/yellow area on the lesion, or on hair, rulers, vignette corners?")

# ============================================================
# DOWNLOAD RESULTS (Colab only)
# ============================================================
try:
    from google.colab import files
    for f in ["gradcam_single.png", "gradcam_panel.png",
              "gradcam_panel_predictions.csv",
              "efficientnet_b0_ham700_seed42.pt"]:
        files.download(f)
except ImportError:
    pass
